# AMRVAC DataConstrain Pipeline

Single-frame workflow: magnetic input -> optional shared vector preprocessing -> optional Grad-Rubin alpha cleaning -> potential field -> one selected NLFFF method (`legacy_mfr`, `optimization`, or `grad_rubin`) -> fixed-boundary DataConstrained MHD. This notebook prepares data and AMRVAC cases; run AMRVAC commands in a terminal.


## 1. Setup

Locate AMRVAC and check the Python environment. **Required for all inputs:** `numpy`, `matplotlib`, and `astropy`. **Additionally required for raw HMI:** `scipy` and `sunpy`. Install them in the environment used by this Jupyter kernel, for example with `%pip install numpy scipy matplotlib astropy sunpy`, then restart the kernel. VTK, tqdm, and DAVE4VM are not required by this single-frame workflow.

**Required:** export `AMRVAC_DIR`. **Optional:** set the fallback below only when that environment variable is unavailable.


In [ ]:
import os
import sys
from pathlib import Path

USER_AMRVAC_DIR = None  # OPTIONAL: set Path('/path/to/amrvac') only if AMRVAC_DIR is not exported.

amrvac_dir = os.environ.get('AMRVAC_DIR') or USER_AMRVAC_DIR
if amrvac_dir is None:
    raise EnvironmentError('Export AMRVAC_DIR before starting Jupyter, or set USER_AMRVAC_DIR above.')
AMRVAC_ROOT = Path(amrvac_dir).expanduser().resolve()
PYTOOLS = AMRVAC_ROOT / 'tools/python'
if not (PYTOOLS / 'amrvac_pytools').is_dir():
    raise ModuleNotFoundError(f'Cannot locate amrvac_pytools under {PYTOOLS}. Check AMRVAC_DIR.')
if str(PYTOOLS) not in sys.path:
    sys.path.insert(0, str(PYTOOLS))

from amrvac_pytools.datadriven import (
    check_data_driven_dependencies, create_notebook_workflow,
    default_notebook_nlfff_run_options, notebook_boundary_options,
    notebook_configuration_report, notebook_single_frame_options,
    normalize_notebook_options,
)

check_data_driven_dependencies()
print(f'AMRVAC root: {AMRVAC_ROOT}')


## 2. Input And Project

Set the magnetic-data directory and inspect the first frame. Raw HMI is shown as $B_{\rm LOS}$ before CEA remapping; CEA input is shown as $B_r$. **Required:** input directory. **Recommended:** project directory and resolution levels. **Optional:** preview limit and block sizes.


In [ ]:
INPUT_DIR = Path('/path/to/raw_HMI')  # REQUIRED
USER_PROJECT_DIR = None  # RECOMMENDED: portable project directory; None uses AMRVAC_ROOT / 'DrivenFieldProject'.
PREVIEW_BMAX = 1000.0  # OPTIONAL: symmetric color limit [G]; None selects it automatically.

# User-facing choices. Both switches are default-off and advisory only.
PREPROCESS_VECTOR = False  # MFR: optional; Optimization/Grad-Rubin: recommended for observations.
PREPROCESSING_MODE = 'recommended'  # none, conservative, or recommended; used only when enabled.
GR_ALPHA_CLEANING = False  # Grad-Rubin observations: recommended, but never auto-enabled.
GR_ALPHA_PRESET = 'recommended'  # none, conservative, or recommended; used only when enabled.
NLFFF_METHOD = 'legacy_mfr'  # Choose exactly one: legacy_mfr, optimization, or grad_rubin.
UNIT_LENGTH_CM = 1.0e9  # AMRVAC unit_length for the selected case.
UNIT_MAGNETICFIELD_G = 100.0  # AMRVAC unit_magneticfield for external Grad-Rubin alpha.
WRITE_DETAILED_METHOD_HISTORY = False  # Optional method-specific logs; common metrics are always produced.

RELAXATION_BOUNDARY_REDUCTION_LEVEL = 3  # Potential/NLFFF boundary reduction.
EVOLUTION_BOUNDARY_REDUCTION_LEVEL = 2  # DataConstrained boundary reduction.
EVOLUTION_AMRVAC_REFINEMENT_LEVEL = 1  # Finest DataConstrained AMR level.
AMRVAC_BLOCK_SIZES = (12, 14, 16, 18, 20)  # Allowed AMRVAC block sizes.

PIPELINE_OPTIONS = normalize_notebook_options(
    nlfff_method=NLFFF_METHOD,
    preprocess_vector=PREPROCESS_VECTOR,
    preprocessing_mode=PREPROCESSING_MODE,
    gr_alpha_cleaning=GR_ALPHA_CLEANING,
    gr_alpha_preset=GR_ALPHA_PRESET,
    unit_length_cm=UNIT_LENGTH_CM,
    unit_magneticfield_g=UNIT_MAGNETICFIELD_G,
)
print(notebook_configuration_report(PIPELINE_OPTIONS))
workflow = create_notebook_workflow(
    workflow_kind='data_constrain',
    amrvac_root=AMRVAC_ROOT,
    project_dir=USER_PROJECT_DIR,
    input_dir=INPUT_DIR,
    relaxation_boundary_reduction_level=RELAXATION_BOUNDARY_REDUCTION_LEVEL,
    evolution_boundary_reduction_level=EVOLUTION_BOUNDARY_REDUCTION_LEVEL,
    evolution_amrvac_refinement_level=EVOLUTION_AMRVAC_REFINEMENT_LEVEL,
    block_sizes=AMRVAC_BLOCK_SIZES,
)
workflow.preview_input(bmax=PREVIEW_BMAX)
print(workflow.input_report())


## 3. Region And Grid Plan

Select one master region. With `REGION_MODE='auto'`, raw HMI uses `RAW_HMI_CEA_PATCH`, while an existing SHARP/CEA map uses `SHARP_PIXEL_WINDOW`; the inactive parameter is ignored. Set `fixed_cea` to force the CEA patch for either input type, or `pixel_window` to force a pixel crop for SHARP/CEA only. Automatic block-compatible trimming is optional.


In [ ]:
REGION_MODE = 'auto'  # RECOMMENDED: auto, fixed_cea, or pixel_window.
RAW_HMI_CEA_PATCH = {  # Active default: auto raw HMI, or fixed_cea with either input type.
    'center_lon': 7.0,
    'center_lat': 13.0,
    'width_degree': 40.0,
    'height_degree': 30.0,
    'resolution_degree': 0.03,
}
SHARP_PIXEL_WINDOW = None  # Used for auto SHARP/CEA or pixel_window; None selects the full map.
# Example crop: {'x0': 100, 'y0': 0, 'nx': 512, 'ny': 512}. Inactive settings are ignored.
AUTO_TRIM_TO_AMRVAC_BLOCKS = True  # OPTIONAL: minimally trim incompatible dimensions when True.

workflow.plan_region(
    region_mode=REGION_MODE,
    raw_hmi_cea_patch=RAW_HMI_CEA_PATCH,
    sharp_pixel_window=SHARP_PIXEL_WINDOW,
    auto_trim=AUTO_TRIM_TO_AMRVAC_BLOCKS,
)
workflow.plot_region(bmax=PREVIEW_BMAX)
print(workflow.region_report())


## 4. Prepare The Reference Frame, Potential, And One NLFFF Case

Prepare only the selected frame, optionally apply shared preprocessing, optionally prepare a Grad-Rubin external-alpha product from that same processed boundary, then stage PotentialField plus exactly one selected NLFFF method. `legacy_mfr` is the existing MHD-embedded magnetofriction path; this notebook does not switch to the independent `mod_magnetofriction` module.


In [ ]:
SNAPSHOT_INDEX = 0  # RECOMMENDED: confirm this when INPUT_DIR contains a sequence; indexing starts at 0.
CEA_REMAP_OPTIONS = {  # OPTIONAL: defaults reproduce the SHARP-like remap.
    'sampling_mode': 'sharp',
    'oversample_resolution_degree': 0.01,
    'smooth_sigma_degree': 0.01,
}
BOUNDARY_NGHOST = 2  # OPTIONAL: ghost cells around the physical magnetogram.
FAIL_ON_PREPROCESS_NONCONVERGENCE = False  # Optional strict mode for preprocessing.
BOUNDARY_VMAX = 500.0  # OPTIONAL: boundary quicklook color limit [G].
POTENTIAL_OPTIONS = {
    'potential_field_method': 'fft',  # RECOMMENDED: fft or green.
    'fft_padding_factor': 2,  # OPTIONAL: FFT horizontal padding.
    'lalpha': 0.0,  # OPTIONAL: 0 gives a potential field; nonzero gives constant-alpha LFFF.
    'fft_top_boundary': 'open',  # OPTIONAL: open or closed.
    # 'potential_zshift_Mm': 3.0,  # OPTIONAL: used only by the Green-function method.
}
NLFFF_RUN_OPTIONS = default_notebook_nlfff_run_options(
    write_detailed_history=WRITE_DETAILED_METHOD_HISTORY,
)
GR_ALPHA_OPTIONS = {
    # 'sigma_Bx': 50.0, 'sigma_By': 50.0, 'sigma_Bz': 20.0,
    # 'resolution_fwhm_pixels': 2.0,
}
BOUNDARY_OPTIONS = notebook_boundary_options(
    PIPELINE_OPTIONS,
    nghost=BOUNDARY_NGHOST,
    fail_on_nonconvergence=FAIL_ON_PREPROCESS_NONCONVERGENCE,
    vmax=BOUNDARY_VMAX,
)
INITIAL_FIELD_OPTIONS = notebook_single_frame_options(
    PIPELINE_OPTIONS,
    snapshot_index=SNAPSHOT_INDEX,
    cea_remap_options=CEA_REMAP_OPTIONS,
    potential_options=POTENTIAL_OPTIONS,
    nlfff_run_options=NLFFF_RUN_OPTIONS,
    gr_alpha_options=GR_ALPHA_OPTIONS,
    boundary_options=BOUNDARY_OPTIONS,
)
workflow.prepare_initial_field(**INITIAL_FIELD_OPTIONS)
print(workflow.initial_field_report())


## 5. Run PotentialField And The Selected NLFFF Method

Run these commands in a terminal, in order. Only the selected NLFFF method is staged. **Recommended:** adjust the MPI process count for the target machine.


In [ ]:
NPROC = 4  # RECOMMENDED: choose a suitable MPI process count for your machine.

print(workflow.initial_field_commands(nproc=NPROC))


## 6. Review Unified NLFFF Metrics And Stage DataConstrained

After the selected NLFFF run finishes, the workflow exposes one common `_nlfff_metrics.csv` path for all three methods. Method-specific detailed CSV files remain disabled unless their explicit detailed-history switch was set above.


In [ ]:
DATA_CONSTRAINED_OPTIONS = {
    'mhd_model': 'zero_beta',  # RECOMMENDED: zero_beta, isothermal, adiabatic, or thermodynamic.
    'atmosphere_model': None,  # OPTIONAL: None chooses uniform/corona/corona/chromosphere by MHD model.
    'atmosphere_source': 'hydrostatic',  # OPTIONAL: hydrostatic or relaxed_table.
    # 'relaxed_atmosphere_file': Path('/path/to/atmosphere.dat'),  # REQUIRED only for relaxed_table.
    # 'coronal_temperature_k': 1.0e6,  # OPTIONAL
    # 'rho_reference_numberdensity_cm3': 1.0e9,  # OPTIONAL
    # 'temperature_curve': 'AL-C7',  # OPTIONAL for chromosphere.
    # 'heating_amplitude_cgs': 1.0e-4,  # OPTIONAL for thermodynamic MHD.
    # 'heating_scale_height_cm': 5.0e9,  # OPTIONAL for thermodynamic MHD.
}

workflow.normalize_initial_nlfff_metrics()
print(workflow.initial_nlfff_metrics_report())
NLFFF_METRICS_PLOT = workflow.plot_initial_nlfff_metrics(show=True)
print('NLFFF metrics quicklook:', NLFFF_METRICS_PLOT['output_path'])

workflow.analyze_and_stage_data_constrained(
    data_constrained_options=DATA_CONSTRAINED_OPTIONS,
)
print(workflow.data_constrained_report())


## 7. Run DataConstrained

Run the staged DataConstrained case in a terminal. Switching `NLFFF_METHOD` requires rerunning the potential/NLFFF staging step so only one NLFFF branch is active.


In [ ]:
print(workflow.data_constrained_commands(nproc=NPROC))
